In [1]:
# Load libraries, display configurations and load the data
import pandas as pd
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

online_retail = pd.read_excel('../data/raw/Online_Retail.xlsx')

In [6]:
# Looking into the table
print(f"Rows: {online_retail.shape[0]:,}")
print(f"Columns: {online_retail.shape[1]}\n")
print(online_retail.head(),"\n")
online_retail.info()

Rows: 541,909
Columns: 8

  invoiceno stockcode                          description  quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          invoicedate  unitprice  customerid         country  
0 2010-12-01 08:26:00       2.55    17850.00  United Kingdom  
1 2010-12-01 08:26:00       3.39    17850.00  United Kingdom  
2 2010-12-01 08:26:00       2.75    17850.00  United Kingdom  
3 2010-12-01 08:26:00       3.39    17850.00  United Kingdom  
4 2010-12-01 08:26:00       3.39    17850.00  United Kingdom   

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------ 

In [5]:
# Changing to lower the columns name
online_retail.columns = online_retail.columns.str.lower()
online_retail.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom


In [8]:
#
print(online_retail['customerid'].dtype)
print(f"Nul values in CustomerID: {online_retail['customerid'].isna().sum():,}")
print(f"Percentage: {online_retail['customerid'].isna().mean()*100:.1f}%")

float64
Nul values in CustomerID: 135,080
Percentage: 24.9%


In [13]:
# looking for wrong values in unitprice
print("Price <=0:")
print(online_retail[online_retail['unitprice'] <= 0]['unitprice'].value_counts())

print("\nCancelaciones (invoice con C):")
cancel = online_retail[online_retail['invoiceno'].astype(str).str.startswith('C')]
print(f"Total: {cancel.shape[0]:,} Rows: {cancel.shape[1]/len(online_retail) * 100:.4f}%")

print("\nNegative Quantity:")
print(online_retail[online_retail['quantity'] < 0].shape[0])

Price <=0:
unitprice
0.00         2515
-11062.06       2
Name: count, dtype: int64

Cancelaciones (invoice con C):
Total: 9,288 Rows: 0.0015%

Negative Quantity:
10624


### Data Quality Assessment — UCI Online Retail

| Issue | Count | % of total | ETL Action |
|---|---|---|---|
| Null CustomerID | 135,080 | 24.9% | Replace with 'UNKNOWN' (can be guest customers) |
| Null Description | 1,454 | 0.3% | Replace with 'No description' |
| Zero/negative UnitPrice | 2,517 | 0.5% | Flag and exclude from revenue |
| Cancelled invoices (C prefix) | 9,288 | 1.7% | Mark as is_return = 1 |
| Negative Quantity | 10,624 | 2.0% | Linked to cancellations |
| CustomerID as float | 541,909 | 100% | Cast to string, remove decimals |

In [16]:
# Confirm if all the negative quantities are cancelations
print(f"Cancelled invoices: {len(cancel):,}")
print(f"Negative quantity rows: {online_retail[online_retail['quantity'] < 0].shape[0]:,}")
print(f"\nMin quantity in cancellations: {cancel['quantity'].min()}")
print(f"All cancellations have negative qty: {(cancel['quantity'] < 0).all()}")


Cancelled invoices: 9,288
Negative quantity rows: 10,624

Min quantity in cancellations: -80995
All cancellations have negative qty: True


We found that all que cancelled orders have negative quantity, but not all the negative quantity in the quantity column, are cancelled orders

In [22]:
# Rows with negative quantity but NOT a cancellation
negative_not_cancelled = online_retail[
    (online_retail['quantity'] < 0) & 
    (~online_retail['invoiceno'].astype(str).str.startswith('C'))
]

print(f"Negative quantity without C prefix: {len(negative_not_cancelled):,}")
print(f"\nSample invoices:")
print(negative_not_cancelled[['invoiceno', 'stockcode', 'description', 'quantity', 'unitprice', 'customerid']].sample(10))
print(f"\nUnique invoice prefixes in this group:")
print(negative_not_cancelled['invoiceno'].astype(str).str[0].value_counts())

Negative quantity without C prefix: 1,336

Sample invoices:
       invoiceno stockcode                 description  quantity  unitprice  \
42564     540010     22501  reverse 21/5/10 adjustment      -100       0.00   
75224     542548     20892                         NaN        -1       0.00   
210963    555334    84805A                 wet damaged       -96       0.00   
514735    579742     85204                      lost??     -1131       0.00   
381675    569874    90180A                 stock check       -32       0.00   
428765    573498     21636                       check       -22       0.00   
132813    547700     21147                         NaN      -105       0.00   
21520     538092     37467                         NaN      -177       0.00   
145256    548886     85118                         NaN        -6       0.00   
514650    579735     21793                       check       -40       0.00   

        customerid  
42564          NaN  
75224          NaN  
210963 

In [23]:
# Check if the unitprice is only 0 or if the negative quantity have another price
print(negative_not_cancelled['unitprice'].nunique())

1


We can consider the 1,336 values that are not canceled invoices as items that were physically removed from warehouse stock (damaged goods, lost inventory, miscounts, etc.). That is why the Quantity column contains negative values and the UnitPrice is 0.

## Final ETL Strategy — Negative Quantity Rows

| Group | Count | Characteristics | Action |
|---|---|---|---|
| Cancelled invoices (C prefix) | 9,288 | Negative qty, valid customer, has description | Keep → mark `is_return = 1` |
| Stock adjustments (no prefix) | 1,336 | Negative qty, price = 0, no customer, no description | Exclude entirely |